# Algoritma Bellman-Ford: notebook sederhana

**Mata kuliah:** Algoritma Komputasi, Magister Teknik Informatika, Universitas Hasanuddin
**Nama / NIM:** Muh Fauzi Ramadhan Nugraha / D082261017

Notebook ini tidak memakai pustaka apa pun: tidak ada satu pun `import`. Jalankan semua sel dari atas ke bawah (Kernel, lalu Restart and Run All).

Isi:
1. Inisiasi
2. Pseudocode dan inti Bellman-Ford
3. Contoh uji tiga kasus kecil (rute kurir, segitiga tukar uang, rantai dengan urutan terburuk)
4. Uji kasus nyata (distribusi barang di Sulawesi Selatan)
5. Kesimpulan

## 1. Inisiasi

Graf disimpan sebagai **daftar sisi**. Simpul diberi nomor 0 sampai n − 1, dan setiap sisi ditulis `(u, v, w)`, artinya ada jalan satu arah dari u ke v dengan bobot (biaya) w. Bobot boleh negatif.

`INF` (tak hingga) dipakai untuk jarak yang belum diketahui. Sel di bawah juga berisi dua fungsi bantu untuk mencetak hasil.

In [1]:
INF = float("inf")          # jarak yang belum diketahui


def tampil(x):
    """Tampilkan jarak: ∞ untuk INF, dan tanda minus yang rapi."""
    return "∞" if x == INF else str(x).replace("-", "−")


def cetak_jejak(nama, jejak):
    """Cetak nilai d setelah tiap putaran sebagai tabel. jejak[0] = keadaan awal."""
    lebar = max(9, max(len(s) for s in nama) + 2)
    print("".ljust(11) + "".join(s.rjust(lebar) for s in nama))
    for k, d in enumerate(jejak):
        label = "Awal" if k == 0 else f"Putaran {k}"
        print(label.ljust(11) + "".join(tampil(x).rjust(lebar) for x in d))


def cetak_rute(nama, dist, pred, src):
    """Cetak jarak dan rute termurah ke tiap simpul (memakai rekonstruksi_jalur di bagian 2)."""
    for v in range(len(nama)):
        if dist[v] == INF:
            print(f"  {nama[v]:<10} tidak terjangkau")
        else:
            rute = " → ".join(nama[i] for i in rekonstruksi_jalur(pred, src, v))
            print(f"  {nama[v]:<10} {tampil(dist[v]):>6}   {rute}")


print("Inisiasi selesai. INF =", INF)

Inisiasi selesai. INF = inf


## 2. Pseudocode dan inti Bellman-Ford

```
BELLMAN-FORD(G, w, s)
1  untuk setiap v ∈ V:  d[v] ← ∞;  π[v] ← NIL
2  d[s] ← 0
3  ulangi |V| − 1 kali:
4      untuk setiap sisi (u, v) ∈ E:
5          jika d[u] < ∞ dan d[u] + w(u, v) < d[v]:
6              d[v] ← d[u] + w(u, v);  π[v] ← u
7  untuk setiap sisi (u, v) ∈ E:
8      jika d[u] < ∞ dan d[u] + w(u, v) < d[v]: kembalikan (d, π, BENAR)
9  kembalikan (d, π, SALAH)
```

Cara membacanya:
- **d[v]** adalah catatan biaya termurah sementara ke v, dan **π[v]** mencatat v didatangi dari simpul mana.
- **Baris 1–2** mengisi nilai awal: sumber 0, simpul lain ∞.
- **Baris 3–6** adalah inti. Semua sisi diperiksa berulang kali, dan setiap kali lewat u lebih murah, catatan v diganti. Langkah ini disebut **relaksasi**.
- **|V| − 1 putaran** selalu cukup, karena jalur termurah yang tidak berputar paling banyak memuat |V| − 1 sisi.
- **Baris 7–8** memeriksa sekali lagi. Kalau masih ada catatan yang bisa turun, ada **siklus negatif**, dan jarak tidak bisa ditentukan.

Kode di bawah mengikuti pseudocode baris demi baris; nomor baris tertulis di komentar. Ada satu tambahan, yaitu **penghentian dini**: kalau satu putaran tidak mengubah apa pun, perulangan berhenti lebih awal. Hasilnya tetap sama.

In [2]:
INF = float("inf")


def inisialisasi(n, src):
    """Pseudocode baris 1-2: semua jarak INF kecuali sumber 0, semua pendahulu None."""
    dist = [INF] * n          # jarak sementara d[v]; INF = belum ada jalur yang diketahui
    pred = [None] * n         # pendahulu pi[v]; None = belum ada
    dist[src] = 0             # jarak sumber ke dirinya sendiri adalah 0
    return dist, pred


def relaksasi(u, v, w, dist, pred):
    """Pseudocode baris 5-6: perbarui d[v] bila lewat u lebih murah. Kembalikan True bila ada pembaruan."""
    # Sisi dari simpul yang belum terjangkau tidak boleh dipakai. Dengan INF float, INF + w tetap INF,
    # tetapi bila INF diganti bilangan besar biasa (mis. 10**9), jarak palsu bisa muncul.
    if dist[u] != INF and dist[u] + w < dist[v]:   # tanda '<' (bukan '<='): nilai sama bukan perbaikan
        dist[v] = dist[u] + w
        pred[v] = u
        return True
    return False


def satu_putaran(edges, dist, pred):
    """Pseudocode baris 4: periksa semua sisi sekali, sesuai urutan di edges.
    dist dan pred diubah di tempat, jadi pembaruan di tengah putaran langsung terlihat oleh sisi berikutnya."""
    ada_perubahan = False
    for u, v, w in edges:
        if relaksasi(u, v, w, dist, pred):
            ada_perubahan = True
    return ada_perubahan


def ada_siklus_negatif(edges, dist):
    """Pseudocode baris 7-8: putaran ke-|V|. Masih ada sisi yang bisa memperbarui d berarti ada siklus
    negatif yang terjangkau dari sumber. Fungsi ini tidak mengubah dist."""
    for u, v, w in edges:
        if dist[u] != INF and dist[u] + w < dist[v]:
            return True
    return False


def bellman_ford(n, edges, src, jejak=None, berhenti_dini=True):
    """Bellman-Ford: jarak terpendek dari satu sumber, bobot negatif diperbolehkan.

    Masukan : n     jumlah simpul (bernomor 0..n-1)
              edges daftar (u, v, w) untuk sisi berarah u -> v berbobot w
              src   simpul sumber
    Keluaran: (dist, pred, ada_siklus_negatif)
              dist[v] jarak terpendek (INF bila tak terjangkau)
              pred[v] simpul sebelum v pada jalur terpendek (None bila tidak ada)
    jejak   : opsional. Bila bukan None, diisi salinan dist: sekali setelah inisialisasi, lalu sekali di
              akhir tiap putaran. Dipakai untuk mencetak tabel jarak per putaran.
    berhenti_dini : bila True, perulangan berhenti begitu satu putaran tidak mengubah apa pun.
    """
    dist, pred = inisialisasi(n, src)                    # baris 1-2
    if jejak is not None:
        jejak.append(dist[:])
    for _ in range(n - 1):                               # baris 3: paling banyak |V| - 1 putaran
        ada_perubahan = satu_putaran(edges, dist, pred)  # baris 4-6
        if jejak is not None:
            jejak.append(dist[:])
        if berhenti_dini and not ada_perubahan:          # penghentian dini (aman: putaran berikutnya identik)
            break
    return dist, pred, ada_siklus_negatif(edges, dist)   # baris 7-9


def rekonstruksi_jalur(pred, src, target):
    """Susun jalur src -> target dengan menelusuri pred mundur. Kembalikan [] bila target tidak terjangkau."""
    jalur = []
    sekarang = target
    langkah = 0
    while sekarang is not None and langkah <= len(pred):   # batas langkah = pengaman bila pred berlingkaran
        jalur.append(sekarang)
        sekarang = pred[sekarang]
        langkah += 1
    if sekarang is not None:                               # berhenti karena batas langkah: pred berlingkaran
        return []
    jalur.reverse()
    return jalur if jalur and jalur[0] == src else []      # simpul tak terjangkau berujung bukan di src

## 3. Contoh uji tiga kasus kecil

Ketiga kasus cukup kecil untuk dihitung dengan tangan. Hasil hitungan tangan ditulis lebih dulu, lalu dicocokkan dengan program memakai `assert`.

### Kasus 1: rute kurir 4 titik

Seorang kurir berangkat dari **Gudang** dan harus sampai ke **Pelanggan**. Ia bisa lewat Toko A, atau mampir dulu ke Toko B. Dari Toko B ke Toko A kurir membawa titipan berbayar, jadi biaya bersih ruas itu **−3** (untung 3).

```
              4
   Gudang ─────────► Toko A ───2───► Pelanggan
      │                 ▲
      5                 │ −3 (titipan berbayar)
      ▼                 │
   Toko B ──────────────┘
```

Urutan sisi yang diperiksa: S1 Gudang→Toko A (4), S2 Gudang→Toko B (5), S3 Toko B→Toko A (−3), S4 Toko A→Pelanggan (2).

**Hitung tangan**

| Langkah | Pemeriksaan | Hasil |
|---|---|---|
| Putaran 1, S1 | 0 + 4 = 4 < ∞ | d[Toko A] = 4 |
| Putaran 1, S2 | 0 + 5 = 5 < ∞ | d[Toko B] = 5 |
| Putaran 1, S3 | 5 − 3 = 2 < 4 | d[Toko A] = 2 |
| Putaran 1, S4 | 2 + 2 = 4 < ∞ | d[Pelanggan] = 4 |
| Putaran 2 | tidak ada yang lebih murah | berhenti |

Hasil: d = (0, 2, 5, 4). Rute termurah ke Pelanggan adalah Gudang → Toko B → Toko A → Pelanggan dengan biaya 4.

In [3]:
NAMA_KURIR = ["Gudang", "Toko A", "Toko B", "Pelanggan"]
SISI_KURIR = [(0, 1, 4), (0, 2, 5), (2, 1, -3), (1, 3, 2)]

jejak = []
dist, pred, siklus = bellman_ford(4, SISI_KURIR, 0, jejak)
cetak_jejak(NAMA_KURIR, jejak)
print("siklus negatif:", siklus)
cetak_rute(NAMA_KURIR, dist, pred, 0)

assert dist == [0, 2, 5, 4]            # sama dengan hitung tangan
assert siklus is False
assert len(jejak) - 1 == 2             # 1 putaran berisi perubahan + 1 putaran konfirmasi
print("Kasus 1 lulus.")

                Gudang     Toko A     Toko B  Pelanggan
Awal                 0          ∞          ∞          ∞
Putaran 1            0          2          5          4
Putaran 2            0          2          5          4
siklus negatif: False
  Gudang          0   Gudang
  Toko A          2   Gudang → Toko B → Toko A
  Toko B          5   Gudang → Toko B
  Pelanggan       4   Gudang → Toko B → Toko A → Pelanggan
Kasus 1 lulus.


**Kenapa Dijkstra salah di sini?** Dijkstra selalu mengambil simpul dengan catatan terkecil, lalu **mengunci** catatan itu. Toko A (4) terkunci sebelum Toko B (5) diproses. Ketika ruas Toko B → Toko A (−3) akhirnya diperiksa, Toko A sudah terkunci, jadi biaya 2 tidak pernah tercatat. Akibatnya Pelanggan ikut salah: 6, padahal seharusnya 4.

Dijkstra di bawah ditulis sederhana tanpa pustaka (memilih simpul terkecil dengan perulangan biasa).

In [4]:
def dijkstra_sederhana(n, edges, src):
    """Dijkstra klasik: simpul dengan catatan terkecil dikunci dan tidak diperbarui lagi."""
    tetangga = [[] for _ in range(n)]
    for u, v, w in edges:
        tetangga[u].append((v, w))
    dist = [INF] * n
    dist[src] = 0
    terkunci = [False] * n
    for _ in range(n):
        u = -1
        for v in range(n):                      # cari simpul belum terkunci dengan catatan terkecil
            if not terkunci[v] and dist[v] < INF and (u == -1 or dist[v] < dist[u]):
                u = v
        if u == -1:
            break
        terkunci[u] = True
        for v, w in tetangga[u]:
            if not terkunci[v] and dist[u] + w < dist[v]:
                dist[v] = dist[u] + w
    return dist


d_dijkstra = dijkstra_sederhana(4, SISI_KURIR, 0)
for i, nama in enumerate(NAMA_KURIR):
    tanda = "" if d_dijkstra[i] == dist[i] else "   <- Dijkstra salah"
    print(f"  {nama:<10} Bellman-Ford {tampil(dist[i]):>3}   Dijkstra {tampil(d_dijkstra[i]):>3}{tanda}")
assert d_dijkstra == [0, 4, 5, 6]

  Gudang     Bellman-Ford   0   Dijkstra   0
  Toko A     Bellman-Ford   2   Dijkstra   4   <- Dijkstra salah
  Toko B     Bellman-Ford   5   Dijkstra   5
  Pelanggan  Bellman-Ford   4   Dijkstra   6   <- Dijkstra salah


### Kasus 2: segitiga ajaib tukar uang

Tiga kurs (angka rekaan):
- 1 USD = 0,9 EUR
- 1 EUR = 160 JPY
- 1 JPY = 0,0072 USD

Kalau 1 USD ditukar keliling USD → EUR → JPY → USD, hasilnya 0,9 × 160 × 0,0072 = **1,0368 USD**. Uangnya bertambah 3,68% hanya dengan berputar. Inilah arbitrase, dan dalam graf ia muncul sebagai **siklus negatif**.

Bellman-Ford menjumlahkan bobot, sedangkan kurs dikalikan. Karena itu bobot ditulis sebagai **−1000·ln(kurs)**, dibulatkan. Perkalian kurs yang lebih dari 1 sama artinya dengan jumlah bobot yang kurang dari 0. Nilainya dihitung lebih dulu dan ditulis langsung, supaya notebook tidak perlu `import math`:

| Sisi | Kurs | Bobot = −1000·ln(kurs) |
|---|---|---|
| USD → EUR | 0,9 | 105 |
| EUR → JPY | 160 | −5075 |
| JPY → USD | 0,0072 | 4934 |
| **Keliling segitiga** | 1,0368 | **−36** |

**Hitung tangan** (sumber USD, |V| − 1 = 2 putaran):

| Putaran | USD | EUR | JPY | Catatan |
|---|---|---|---|---|
| Awal | 0 | ∞ | ∞ | |
| 1 | −36 | 105 | −4970 | JPY → USD: −4970 + 4934 = −36 < 0, catatan sumber sendiri turun |
| 2 | −72 | 69 | −5006 | turun lagi sebesar 36 |
| Uji (baris 7–8) | | | | USD → EUR: −72 + 105 = 33 < 69, masih bisa turun: **siklus negatif** |

In [5]:
print("hasil keliling 1 USD:", 0.9 * 160 * 0.0072, "USD")

NAMA_UANG = ["USD", "EUR", "JPY"]
SISI_UANG = [(0, 1, 105), (1, 2, -5075), (2, 0, 4934)]     # bobot = -1000 * ln(kurs), dibulatkan

jejak = []
dist, pred, siklus = bellman_ford(3, SISI_UANG, 0, jejak)
cetak_jejak(NAMA_UANG, jejak)
print("siklus negatif:", siklus)

assert siklus is True                   # arbitrase terdeteksi
assert jejak[1] == [-36, 105, -4970]    # sama dengan hitung tangan putaran 1
assert jejak[2] == [-72, 69, -5006]     # dan putaran 2
print("Kasus 2 lulus: ada peluang untung dari berputar, jadi jarak terpendek tidak terdefinisi.")

hasil keliling 1 USD: 1.0368 USD
                 USD      EUR      JPY
Awal               0        ∞        ∞
Putaran 1        −36      105    −4970
Putaran 2        −72       69    −5006
siklus negatif: True
Kasus 2 lulus: ada peluang untung dari berputar, jadi jarak terpendek tidak terdefinisi.


### Kasus 3: rantai pengantaran dengan urutan terburuk

Rutenya lurus: A → B → C → D dengan biaya 2, 3, dan 1. Jawabannya jelas d = (0, 2, 5, 6). Yang diuji di sini adalah **urutan daftar sisi**. Bellman-Ford memeriksa sisi menurut urutan daftar, dan urutan itu menentukan berapa putaran yang dibutuhkan.

- **Daftar searah** (A→B, B→C, C→D): satu putaran sudah mengisi semuanya.
- **Daftar terbalik** (C→D, B→C, A→B): di tiap putaran hanya satu simpul yang maju, karena sisi berikutnya diperiksa sebelum titik asalnya punya catatan.

**Hitung tangan, daftar terbalik**

| Putaran | A | B | C | D | Yang berubah |
|---|---|---|---|---|---|
| Awal | 0 | ∞ | ∞ | ∞ | |
| 1 | 0 | 2 | ∞ | ∞ | C→D dan B→C dilewati (asal masih ∞), A→B mengisi B |
| 2 | 0 | 2 | 5 | ∞ | B→C mengisi C |
| 3 | 0 | 2 | 5 | 6 | C→D mengisi D |

Daftar terbalik memakai seluruh |V| − 1 = 3 putaran: 3 putaran × 3 sisi + 3 uji siklus = **12 pemeriksaan**. Daftar searah: 1 putaran berisi perubahan + 1 konfirmasi + 3 uji = **9 pemeriksaan**. Pada graf besar perbedaan ini menjadi |V| putaran lawan beberapa putaran saja, dan itulah sumber kasus terburuk Θ(|V|·|E|).

In [6]:
NAMA_RANTAI = ["A", "B", "C", "D"]
SEARAH = [(0, 1, 2), (1, 2, 3), (2, 3, 1)]
TERBALIK = [(2, 3, 1), (1, 2, 3), (0, 1, 2)]


def jumlah_pemeriksaan(jejak, m):
    """Putaran yang dijalankan x jumlah sisi, ditambah satu kali uji siklus negatif."""
    return (len(jejak) - 1) * m + m


hasil = {}
for nama_urutan, daftar in (("searah", SEARAH), ("terbalik", TERBALIK)):
    jejak = []
    dist, pred, siklus = bellman_ford(4, daftar, 0, jejak)
    hasil[nama_urutan] = (dist, len(jejak) - 1, jumlah_pemeriksaan(jejak, len(daftar)))
    print(f"Daftar {nama_urutan}: {len(jejak) - 1} putaran dijalankan, {jumlah_pemeriksaan(jejak, len(daftar))} pemeriksaan sisi")
    cetak_jejak(NAMA_RANTAI, jejak)
    print()

assert hasil["searah"][0] == hasil["terbalik"][0] == [0, 2, 5, 6]   # jawaban sama
assert hasil["searah"][1:] == (2, 9)
assert hasil["terbalik"][1:] == (3, 12)
print("Kasus 3 lulus: urutan sisi mengubah jumlah kerja, tidak mengubah jawaban.")

Daftar searah: 2 putaran dijalankan, 9 pemeriksaan sisi
                   A        B        C        D
Awal               0        ∞        ∞        ∞
Putaran 1          0        2        5        6
Putaran 2          0        2        5        6

Daftar terbalik: 3 putaran dijalankan, 12 pemeriksaan sisi
                   A        B        C        D
Awal               0        ∞        ∞        ∞
Putaran 1          0        2        ∞        ∞
Putaran 2          0        2        5        ∞
Putaran 3          0        2        5        6

Kasus 3 lulus: urutan sisi mengubah jumlah kerja, tidak mengubah jawaban.


## 4. Uji kasus nyata: distribusi barang di Sulawesi Selatan

Sebuah ekspedisi (rekaan) mengirim barang dari **Makassar** ke tujuh kota lain. Biaya bersih tiap ruas adalah biaya operasional dikurangi pendapatan muatan balik, dalam ribu rupiah per ton. Dua ruas bernilai negatif: R1 Pinrang→Sidrap (50 − 80 = −30) dan R2 Pangkep→Barru (60 − 80 = −20). Semua angka rekaan; nama kota hanya memberi konteks.

Studi kasus ini menguji jarak dengan bobot negatif pada graf yang lebih besar daripada kasus kecil. Siklus negatif sudah ditunjukkan oleh segitiga tukar uang (kasus 2), sedangkan simpul yang tidak terjangkau diperiksa lewat uji silang di notebook eksperimen.

In [7]:
NAMA_KOTA = ["Makassar", "Maros", "Pangkep", "Barru", "Parepare", "Sidrap", "Pinrang", "Enrekang"]

# (kode, dari, ke, biaya operasional, pendapatan muatan); urutan baris = urutan sisi diperiksa
RUAS = [
    ("R1", 6, 5, 50, 80),    # Pinrang  -> Sidrap
    ("R2", 2, 3, 60, 80),    # Pangkep  -> Barru
    ("R3", 0, 2, 100, 0),    # Makassar -> Pangkep
    ("R4", 3, 4, 75, 0),     # Barru    -> Parepare
    ("R5", 5, 6, 45, 0),     # Sidrap   -> Pinrang
    ("R6", 0, 1, 40, 0),     # Makassar -> Maros
    ("R7", 5, 7, 70, 0),     # Sidrap   -> Enrekang
    ("R8", 4, 6, 60, 0),     # Parepare -> Pinrang
    ("R9", 1, 2, 50, 0),     # Maros    -> Pangkep
    ("R10", 4, 5, 55, 0),    # Parepare -> Sidrap
]


def bangun_studi_kasus():
    """Kembalikan daftar sisi studi kasus. Bobot = biaya - pendapatan."""
    return [(u, v, biaya - pendapatan) for kode, u, v, biaya, pendapatan in RUAS]


HARAPAN = [0, 40, 90, 70, 145, 175, 205, 245]
edges = bangun_studi_kasus()
dist, pred, siklus = bellman_ford(8, edges, 0)
print("siklus negatif:", siklus)
cetak_rute(NAMA_KOTA, dist, pred, 0)
assert not siklus and dist == HARAPAN
print("Studi kasus lulus.")

siklus negatif: False
  Makassar        0   Makassar
  Maros          40   Makassar → Maros
  Pangkep        90   Makassar → Maros → Pangkep
  Barru          70   Makassar → Maros → Pangkep → Barru
  Parepare      145   Makassar → Maros → Pangkep → Barru → Parepare
  Sidrap        175   Makassar → Maros → Pangkep → Barru → Parepare → Pinrang → Sidrap
  Pinrang       205   Makassar → Maros → Pangkep → Barru → Parepare → Pinrang
  Enrekang      245   Makassar → Maros → Pangkep → Barru → Parepare → Pinrang → Sidrap → Enrekang
Studi kasus lulus.


Jejak studi kasus dan perbandingan dengan Dijkstra. Sidrap baru turun dari 200 ke 175 pada putaran 3, lewat ruas R1 yang bernilai negatif. Dijkstra sudah mengunci Sidrap pada 200 sebelum itu.

In [8]:
edges = bangun_studi_kasus()
jejak = []
dist, pred, siklus = bellman_ford(8, edges, 0, jejak)
cetak_jejak(NAMA_KOTA, jejak)

d_dijkstra = dijkstra_sederhana(8, edges, 0)
print()
for i, nama in enumerate(NAMA_KOTA):
    tanda = "" if d_dijkstra[i] == dist[i] else "   <- Dijkstra salah"
    print(f"  {nama:<10} Bellman-Ford {tampil(dist[i]):>4}   Dijkstra {tampil(d_dijkstra[i]):>4}{tanda}")
assert d_dijkstra[5] == 200 and dist[5] == 175

             Makassar     Maros   Pangkep     Barru  Parepare    Sidrap   Pinrang  Enrekang
Awal                0         ∞         ∞         ∞         ∞         ∞         ∞         ∞
Putaran 1           0        40        90         ∞         ∞         ∞         ∞         ∞
Putaran 2           0        40        90        70       145       200       205         ∞
Putaran 3           0        40        90        70       145       175       205       245
Putaran 4           0        40        90        70       145       175       205       245

  Makassar   Bellman-Ford    0   Dijkstra    0
  Maros      Bellman-Ford   40   Dijkstra   40
  Pangkep    Bellman-Ford   90   Dijkstra   90
  Barru      Bellman-Ford   70   Dijkstra   70
  Parepare   Bellman-Ford  145   Dijkstra  145
  Sidrap     Bellman-Ford  175   Dijkstra  200   <- Dijkstra salah
  Pinrang    Bellman-Ford  205   Dijkstra  205
  Enrekang   Bellman-Ford  245   Dijkstra  270   <- Dijkstra salah
